In [12]:
import os
from database.db import DatabaseTX
from database.db import Record
from database.ql import QLExecutor

In [13]:
os.system("rm -rf test.mydb")

0

In [14]:
db = DatabaseTX("test.mydb")

In [15]:
db.open()

In [16]:
executor = QLExecutor(db)

In [17]:
text = """
CREATE TABLE users (
    id INTEGER,
    name STRING,
    age INTEGER,
    height INTEGER,
    INDEX (name),
    INDEX (age),
    INDEX (height),
    PRIMARY KEY (id)
);

INSERT INTO users (id, name, age, height) VALUES (1, 'Alice', 25, 165 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (2, 'Bob', 30, 175 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (3, 'Cindy', 35, 185 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (4, 'David', 40, 195 - 50 % 32);

SELECT id AS user_id, name, age, height FROM users INDEX BY name >= 'Bob' AND name <= 'Dob' FILTER age <= 40 LIMIT 10;
"""

In [18]:
recs = executor.execute(text)
recs

[Record(user_id=2, name=b'Bob', age=30, height=157),
 Record(user_id=3, name=b'Cindy', age=35, height=167),
 Record(user_id=4, name=b'David', age=40, height=177)]

In [19]:
print(Record.format_records(recs))

# 3 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 2          | Bob        | 30         | 157       |
----------------------------------------------------
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [20]:
print(Record.format_records(executor.execute("""
UPDATE users SET age = 50 INDEX BY name >= 'Bob' AND name <= 'Bob';
SELECT id AS user_id, name, age, height FROM users INDEX BY name >= 'Bob' AND name <= 'Dob' FILTER age <= 40 LIMIT 10;
""")))

# 2 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [21]:
print(Record.format_records(executor.execute("""
DELETE FROM users INDEX BY name >= 'David' AND name <= 'David';
SELECT id AS user_id, name, age, height FROM users INDEX BY name >= 'Bob' AND name <= 'Dob' FILTER age <= 40 LIMIT 10;
""")))

# 1 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------



In [22]:
db.close()